## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import java.io.*;
import java.util.*;

public class Main {
    
   
    static class MagicBoard {
        int n, targetA, targetB, lowerBitLimit;
        int[] stones;
        int[] positions;
        List<Integer> actions;

        public MagicBoard(int n, int a, int b, int[] initialStones) {
            this.n = n;
            this.targetA = a;
            this.targetB = b;
            this.stones = initialStones;
            this.positions = new int[n];
            this.actions = new ArrayList<>();
            refreshPositions();
        }

        
        void refreshPositions() {
            for (int i = 0; i < n; i++) {
                positions[stones[i]] = i;
            }
        }

        
        void performSwap() {
            actions.add(0);
            for (int i = 0; i < n; i++) {
                if (stones[i] == targetA) stones[i] = targetB;
                else if (stones[i] == targetB) stones[i] = targetA;
            }
            refreshPositions();
        }

        
        void performAdd(int val) {
            if (val == 0) return;
            actions.add(val);
            for (int i = 0; i < n; i++) {
                stones[i] = (stones[i] + val) % n;
            }
            refreshPositions();
        }

        
        void performXor(int val) {
            if (val == 0) return;
            actions.add(-val);
            for (int i = 0; i < n; i++) {
                stones[i] ^= val;
            }
            refreshPositions();
        }

        
        int[] calculatePath(int startVal, int endVal) {
            int diff = (endVal - startVal + n - lowerBitLimit + n) % n;
            int pa = 0, pb = 0;
            for (int s = n / 2; s >= 2 * lowerBitLimit; s /= 2) {
                if (diff >= s) {
                    diff -= s;
                    pb += s / 2;
                } else {
                    pa += s / 2;
                }
            }
            pa += n / 2 + (startVal & (lowerBitLimit - 1));
            pb += (startVal & (lowerBitLimit - 1));
            return new int[]{pa, pb};
        }

        
        void exchangePair(int idx1, int idx2) {
            if ((idx1 / lowerBitLimit) % 2 == (idx2 / lowerBitLimit) % 2) {
                int temp;
                if ((idx1 / lowerBitLimit) % 2 == 0) {
                    temp = (idx1 & (lowerBitLimit - 1)) + lowerBitLimit;
                } else {
                    temp = (idx1 & (lowerBitLimit - 1));
                }
                exchangePair(idx1, temp);
                exchangePair(idx2, temp);
                exchangePair(idx1, temp);
            } else {
                int pa = calculatePath(targetA, targetB)[0];
                int pc = calculatePath(idx1, idx2)[0];

                performAdd((pc - idx1 + n) % n);
                performXor(pc ^ pa);
                performAdd((targetA - pa + n) % n);
                performSwap();
                performAdd((pa - targetA + n) % n);
                performXor(pc ^ pa);
                performAdd((idx1 - pc + n) % n);
            }
        }

        
        void solve() {
            lowerBitLimit = (targetA - targetB + n) % n;
            lowerBitLimit = lowerBitLimit & -lowerBitLimit;
            if (lowerBitLimit == 0) {
                lowerBitLimit = n;
            }

            
            if (lowerBitLimit > 1) {
                ResidueBuilder builder = new ResidueBuilder(lowerBitLimit);
                for (int i = 0; i < n; i++) {
                    builder.sequence[i] = stones[i] & (lowerBitLimit - 1);
                }
                if (!builder.build()) {
                    System.out.println("-1");
                    System.exit(0);
                }
                for (int move : builder.moves) {
                    if (move > 0) performAdd(move);
                    else performXor(-move);
                }
            }

            
            for (int r = 0; r < lowerBitLimit; r++) {
                List<Integer> bucket = new ArrayList<>();
                for (int j = r; j < n; j += lowerBitLimit) {
                    bucket.add(stones[j]);
                }
                Collections.sort(bucket);
                int bIdx = 0;
                for (int j = r; j < n; j += lowerBitLimit) {
                    if (bucket.get(bIdx) != j) {
                        System.out.println("-1");
                        System.exit(0);
                    }
                    bIdx++;
                }

                for (int j = r; j < n; j += lowerBitLimit) {
                    if (stones[j] != j) {
                        exchangePair(j, stones[j]);
                    }
                }
            }
        }
    }

    
    static class ResidueBuilder {
        int size;
        int[] sequence = new int[1024];
        List<Integer> moves = new ArrayList<>();

        ResidueBuilder(int size) {
            this.size = size;
        }

        boolean build() {
            if (size == 1) return true;

            ResidueBuilder even = new ResidueBuilder(size / 2);
            ResidueBuilder odd = new ResidueBuilder(size / 2);

            for (int i = 0; i < size / 2; i++) {
                even.sequence[i] = sequence[2 * i] / 2;
                odd.sequence[i] = sequence[2 * i + 1] / 2;
            }

            if (!even.build() || !odd.build()) return false;

            if (sequence[0] % 2 != 0) {
                moves.add(size == 2 ? 1 : -1);
            }

            int sumEven = 0, sumOdd = 0;
            for (int x : even.moves) {
                if (x > 0) {
                    moves.add(-1); moves.add(1);
                } else {
                    moves.add(2 * x);
                    sumEven ^= (-2 * x);
                }
            }
            if (sumEven != 0) moves.add(-sumEven);

            for (int x : odd.moves) {
                if (x > 0) {
                    moves.add(1); moves.add(-1);
                } else {
                    moves.add(2 * x);
                    sumOdd ^= (-2 * x);
                }
            }

            if ((sumOdd & (size / 2)) != (sumEven & (size / 2))) {
                for (int i = 0; i < size / 4; i++) {
                    moves.add(-1); moves.add(1);
                }
            }

            if (sumEven >= size / 2) sumEven -= size / 2;
            if (sumOdd >= size / 2) sumOdd -= size / 2;

            if (sumEven != sumOdd) return false;

           
            List<Integer> compressed = new ArrayList<>();
            for (int x : moves) {
                if (compressed.isEmpty()) compressed.add(x);
                else if (x < 0 && compressed.get(compressed.size() - 1) < 0) {
                    int last = compressed.remove(compressed.size() - 1);
                    int merged = -((-last) ^ (-x));
                    if (merged != 0) compressed.add(merged);
                } else {
                    compressed.add(x);
                }
            }
            moves = compressed;
            return true;
        }
    }

    public static void main(String[] args) throws IOException {
       
        FastReader in = new FastReader();
        int n = in.nextInt();
        if (n == -1) return;
        
        int a = in.nextInt();
        int b = in.nextInt();
        int[] initialStones = new int[n];
        for (int i = 0; i < n; i++) {
            initialStones[i] = in.nextInt();
        }

        MagicBoard board = new MagicBoard(n, a, b, initialStones);
        board.solve();

        PrintWriter out = new PrintWriter(System.out);
        out.println(board.actions.size());
        for (int action : board.actions) {
            if (action == 0) out.println("0");
            else if (action < 0) out.println("1 " + (-action));
            else out.println("2 " + action);
        }
        out.flush(); 
    }

    
    static class FastReader {
        BufferedReader br;
        StringTokenizer st;

        public FastReader() {
            br = new BufferedReader(new InputStreamReader(System.in));
        }

        String next() {
            while (st == null || !st.hasMoreElements()) {
                try {
                    String line = br.readLine();
                    if (line == null) return null;
                    st = new StringTokenizer(line);
                } catch (IOException e) {
                    e.printStackTrace();
                }
            }
            return st.nextToken();
        }

        int nextInt() {
            String s = next();
            if (s == null) return -1;
            return Integer.parseInt(s);
        }
    }
}

## B 长跑

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

void solve() {
    int N, L, Maxn, S;
    
    while (cin >> N >> L >> Maxn >> S) {
        vector<pair<int, int>> stations;
        for (int i = 0; i < N; ++i) {
            int p, c;
            cin >> p >> c;
            
            if (p > 0 && p < L) {
                stations.push_back({p, c});
            }
        }

        
        if (Maxn >= L) {
            cout << "Yes\n";
            continue;
        }

       
        sort(stations.begin(), stations.end());

        int n = stations.size();
        vector<int> dp(n, 1e9); 
        bool possible = false;

       
        for (int i = 0; i < n; ++i) {
            
            if (stations[i].first <= Maxn) {
                dp[i] = stations[i].second;
            }
            
            
            for (int j = 0; j < i; ++j) {
                
                if (stations[i].first - stations[j].first <= Maxn) {
                    dp[i] = min(dp[i], dp[j] + stations[i].second);
                }
            }
            
            
            if (L - stations[i].first <= Maxn && dp[i] <= S) {
                possible = true;
                break; 
            }
        }

        
        if (possible) {
            cout << "Yes\n";
        } else {
            cout << "No\n";
        }
    }
}

int main() {
    
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    solve();
    
    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

typedef unsigned long long ull;

const int MAXN = 100005;
ull P1[MAXN], P2[MAXN];
ull H1_A[MAXN], H2_A[MAXN];
ull H1_B[MAXN], H2_B[MAXN];

const ull B1 = 313;
const ull B2 = 317;
const ull M1 = 1e9 + 7;
const ull M2 = 1e9 + 9;


void init_hash(int n, const string& revA, const string& B) {
    P1[0] = 1; P2[0] = 1;
    for (int i = 1; i <= n; i++) {
        P1[i] = (P1[i - 1] * B1) % M1;
        P2[i] = (P2[i - 1] * B2) % M2;
    }
    
    H1_A[0] = 0; H2_A[0] = 0;
    for (int i = 1; i <= n; i++) {
        H1_A[i] = (H1_A[i - 1] * B1 + revA[i - 1]) % M1;
        H2_A[i] = (H2_A[i - 1] * B2 + revA[i - 1]) % M2;
    }
    
    H1_B[0] = 0; H2_B[0] = 0;
    for (int i = 1; i <= n; i++) {
        H1_B[i] = (H1_B[i - 1] * B1 + B[i - 1]) % M1;
        H2_B[i] = (H2_B[i - 1] * B2 + B[i - 1]) % M2;
    }
}


pair<ull, ull> get_hash_A(int L, int R) {
    ull h1 = (H1_A[R] + M1 - (H1_A[L - 1] * P1[R - L + 1]) % M1) % M1;
    ull h2 = (H2_A[R] + M2 - (H2_A[L - 1] * P2[R - L + 1]) % M2) % M2;
    return {h1, h2};
}


pair<ull, ull> get_hash_B(int L, int R) {
    ull h1 = (H1_B[R] + M1 - (H1_B[L - 1] * P1[R - L + 1]) % M1) % M1;
    ull h2 = (H2_B[R] + M2 - (H2_B[L - 1] * P2[R - L + 1]) % M2) % M2;
    return {h1, h2};
}


int get_lcp(int idxA, int idxB, int n) {
    int low = 0, high = n - max(idxA, idxB) + 1;
    int ans = 0;
    while (low <= high) {
        int mid = low + (high - low) / 2;
        if (mid == 0) {
            ans = 0;
            low = mid + 1;
            continue;
        }
        if (get_hash_A(idxA, idxA + mid - 1) == get_hash_B(idxB, idxB + mid - 1)) {
            ans = mid;
            low = mid + 1;
        } else {
            high = mid - 1;
        }
    }
    return ans;
}


vector<int> manacher(const string& str) {
    string S = "#";
    for (char c : str) {
        S += c;
        S += '#';
    }
    int m = S.length();
    vector<int> rad(m, 0);
    int center = 0, right = 0;
    for (int i = 0; i < m; i++) {
        if (i < right) {
            rad[i] = min(right - i, rad[2 * center - i]);
        }
        while (i - rad[i] - 1 >= 0 && i + rad[i] + 1 < m && S[i - rad[i] - 1] == S[i + rad[i] + 1]) {
            rad[i]++;
        }
        if (i + rad[i] > right) {
            center = i;
            right = i + rad[i];
        }
    }
    return rad;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int n;
    if (!(cin >> n)) return 0;
    
    string A, B;
    cin >> A >> B;
    
    string revA = A;
    reverse(revA.begin(), revA.end());
    
    init_hash(n, revA, B);
    
    vector<int> radA = manacher(A);
    vector<int> radB = manacher(B);
    
    int ans = 0;
    
    
    for (int i = 0; i < (int)radA.size(); i++) {
        if (radA[i] > 0) {
            int L = (i - radA[i]) / 2 + 1; 
            int R = (i + radA[i]) / 2;     
            int len = radA[i];
            int ext = 0;
            if (L > 1 && R <= n) {
                int idxA = n - L + 2;
                int idxB = R;
                ext = get_lcp(idxA, idxB, n);
            }
            ans = max(ans, len + 2 * ext);
        }
    }
    
    
    for (int i = 0; i < (int)radB.size(); i++) {
        if (radB[i] > 0) {
            int L = (i - radB[i]) / 2 + 1;
            int R = (i + radB[i]) / 2;
            int len = radB[i];
            int ext = 0;
            if (L >= 1 && R + 1 <= n) {
                int idxA = n - L + 1;
                int idxB = R + 1;
                ext = get_lcp(idxA, idxB, n);
            }
            ans = max(ans, len + 2 * ext);
        }
    }
    
    
    for (int i = 1; i <= n; i++) {
        int idxA = n - i + 1;
        int idxB = i;
        int ext = get_lcp(idxA, idxB, n);
        ans = max(ans, 2 * ext);
    }
    
    cout << ans << "\n";
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>

using namespace std;

const int MAX_X = 100005;

int last_pos_arr[MAX_X];
int state_arr[MAX_X];

struct Op {
    int type; 
    int x;
};

void solve() {
    int m;
    
    while (cin >> m) {
        if (m == 0) {
            cout << -1 << "\n";
            continue;
        }

        vector<Op> ops(m + 1);
        vector<int> parent(m + 2);

        
        for (int i = 1; i <= m; ++i) {
            string s;
            cin >> s;
            
            if (s[0] == 'I') {
                int x; cin >> x;
                ops[i] = {1, x};
                parent[i] = i + 1;
            } else if (s[0] == 'O') {
                int x; cin >> x;
                ops[i] = {2, x};
                parent[i] = i + 1;
            } else {
                ops[i] = {0, 0};
                parent[i] = i; 
            }
        }
        parent[m + 1] = m + 1; 

        
        auto find_set = [&](auto& self, int i) -> int {
            if (parent[i] == i) return i;
            return parent[i] = self(self, parent[i]);
        };

        vector<int> touched; 
        int error_line = -1;

       
        for (int i = 1; i <= m; ++i) {
            if (ops[i].type == 0) continue; 

            int type = ops[i].type;
            int x = ops[i].x;
            touched.push_back(x);

            if (type == 1) { 
                if (state_arr[x] == 1) {
                    
                    int p = find_set(find_set, last_pos_arr[x] + 1);
                    if (p < i) { 
                        parent[p] = p + 1; 
                        last_pos_arr[x] = i; 
                    } else {
                        error_line = i;
                        break;
                    }
                } else {
                    state_arr[x] = 1;
                    last_pos_arr[x] = i;
                }
            } else if (type == 2) { 
                if (state_arr[x] == 0) {
                   
                    int p = find_set(find_set, last_pos_arr[x] + 1);
                    if (p < i) { 
                        parent[p] = p + 1; 
                        last_pos_arr[x] = i;
                    } else {
                        error_line = i;
                        break;
                    }
                } else {
                    state_arr[x] = 0;
                    last_pos_arr[x] = i;
                }
            }
        }

        
        if (error_line != -1) {
            cout << error_line << "\n";
        } else {
            cout << -1 << "\n";
        }

        
        for (int x : touched) {
            last_pos_arr[x] = 0;
            state_arr[x] = 0;
        }
    }
}

int main() {
    
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    solve();
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>

using namespace std;


struct Point {
    int x, y;
};

int main() {
    
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    if (!(cin >> n)) return 0;

    vector<Point> points(n);
    for (int i = 0; i < n; ++i) {
        cin >> points[i].x >> points[i].y;
    }

    vector<bool> visited(n, false);
    int components = 0; 

    
    auto dfs = [&](auto& self, int u) -> void {
        visited[u] = true;
        for (int v = 0; v < n; ++v) {
            if (!visited[v]) {
                
                if (points[u].x == points[v].x || points[u].y == points[v].y) {
                    self(self, v);
                }
            }
        }
    };

   
    for (int i = 0; i < n; ++i) {
        if (!visited[i]) {
            components++; 
            dfs(dfs, i);  
        }
    }

    
    cout << components - 1 << "\n";

    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <vector>
#include <cstring>

using namespace std;


using ull = unsigned long long;
const ull BASE1 = 13331;
const ull BASE2 = 23333;
const ull MOD = 1000000007ULL;
const int MAX_LEN = 100005;

ull p_pow1[MAX_LEN], p_pow2[MAX_LEN];
ull h1_S[MAX_LEN], h2_S[MAX_LEN];

int8_t dp[MAX_LEN];
int8_t next_dp[MAX_LEN];

enum TokenType { STAR, QUESTION, EXACT };
struct Token {
    TokenType type;
    ull h1;
    ull h2;
    int len;
};

void solve() {
    string pattern;
    
    while (cin >> pattern) {
        int n;
        if (!(cin >> n)) break;

       
        vector<Token> tokens;
        string curr = "";
        for (char c : pattern) {
            if (c == '*' || c == '?') {
                if (!curr.empty()) {
                    ull h1 = 0, h2 = 0;
                    for (char ch : curr) {
                        h1 = h1 * BASE1 + ch;
                        h2 = (h2 * BASE2 + ch) % MOD;
                    }
                    tokens.push_back({EXACT, h1, h2, (int)curr.length()});
                    curr = "";
                }
                if (c == '*') {
                    if (tokens.empty() || tokens.back().type != STAR) {
                        tokens.push_back({STAR, 0, 0, 0});
                    }
                } else {
                    tokens.push_back({QUESTION, 0, 0, 0});
                }
            } else {
                curr += c;
            }
        }
        if (!curr.empty()) {
            ull h1 = 0, h2 = 0;
            for (char ch : curr) {
                h1 = h1 * BASE1 + ch;
                h2 = (h2 * BASE2 + ch) % MOD;
            }
            tokens.push_back({EXACT, h1, h2, (int)curr.length()});
        }

       
        int min_req = 0;
        for (auto& tok : tokens) {
            if (tok.type == EXACT) min_req += tok.len;
            else if (tok.type == QUESTION) min_req += 1;
        }

        int T = tokens.size();

        
        for (int idx = 0; idx < n; ++idx) {
            string S;
            cin >> S;
            int N = S.length();

            if (N < min_req) {
                cout << "NO\n";
                continue;
            }

           
            for (int i = 0; i < N; ++i) {
                h1_S[i + 1] = h1_S[i] * BASE1 + S[i];
                h2_S[i + 1] = (h2_S[i] * BASE2 + S[i]) % MOD;
            }

            auto get_hash1 = [&](int l, int r) {
                return h1_S[r] - h1_S[l] * p_pow1[r - l];
            };
            auto get_hash2 = [&](int l, int r) {
                return (h2_S[r] + MOD - (h2_S[l] * p_pow2[r - l]) % MOD) % MOD;
            };

            
            memset(dp, 0, N + 1);
            dp[N] = 1; 
            for (int t = T - 1; t >= 0; --t) {
                memset(next_dp, 0, N + 1);
                
                if (tokens[t].type == STAR) {
                    int8_t has_true = 0;
                    for (int i = N; i >= 0; --i) {
                        has_true |= dp[i]; 
                        next_dp[i] = has_true;
                    }
                } else if (tokens[t].type == QUESTION) {
                    for (int i = N - 1; i >= 0; --i) {
                        next_dp[i] = dp[i + 1]; 
                    }
                } else {
                    int len = tokens[t].len;
                    ull h1_tok = tokens[t].h1;
                    ull h2_tok = tokens[t].h2;
                    for (int i = N - len; i >= 0; --i) {
                        
                        if (get_hash1(i, i + len) == h1_tok && get_hash2(i, i + len) == h2_tok) {
                            next_dp[i] = dp[i + len];
                        }
                    }
                }
                
                memcpy(dp, next_dp, N + 1);
            }

            
            if (dp[0]) cout << "YES\n";
            else cout << "NO\n";
        }
    }
}

int main() {
    
    ios::sync_with_stdio(false);
    cin.tie(nullptr);


    p_pow1[0] = 1;
    p_pow2[0] = 1;
    for (int i = 1; i < MAX_LEN; ++i) {
        p_pow1[i] = p_pow1[i - 1] * BASE1;
        p_pow2[i] = (p_pow2[i - 1] * BASE2) % MOD;
    }

    solve();
    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <vector>

using namespace std;


long long f[35][3];
int t[35][3];
pair<int, int> prio[6];

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    if (!(cin >> n)) return 0;

    for (int i = 0; i < 6; ++i) {
        string s;
        cin >> s;
        prio[i] = {s[0] - 'A', s[1] - 'A'};
    }

    
    for (int x = 0; x < 3; ++x) {
        for (int i = 0; i < 6; ++i) {
            if (prio[i].first == x) {
                t[1][x] = prio[i].second;
                f[1][x] = 1;
                break;
            }
        }
    }

    
    for (int i = 2; i <= n; ++i) {
        for (int x = 0; x < 3; ++x) {
            int y = t[i - 1][x];    
            int z = 3 - x - y;      
            
            
            if (t[i - 1][y] == z) {
                f[i][x] = f[i - 1][x] + 1 + f[i - 1][y];
                t[i][x] = z;
            } 
            
            else {
                f[i][x] = f[i - 1][x] + 1 + f[i - 1][y] + 1 + f[i - 1][x];
                t[i][x] = y;
            }
        }
    }

    
    cout << f[n][0] << endl;

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <iostream>
#include <algorithm>
#include <cmath>

using namespace std;

void solve() {
    long long xp, yp, xs, ys;
    
    if (!(cin >> xp >> yp >> xs >> ys)) return;

    
    long long dx = abs(xp - xs);
    long long dy = abs(yp - ys);

    
    if (dx < dy) {
        swap(dx, dy);
    }

    
    if (dx == 1 && dy == 0) {
        cout << 3 << "\n";
        return;
    }
    if (dx == 2 && dy == 2) {
        cout << 4 << "\n";
        return;
    }

    
    long long res = max((dx + 1) / 2, (dx + dy + 2) / 3);

    
    if ((res % 2) != ((dx + dy) % 2)) {
        res++;
    }

    cout << res << "\n";
}

int main() {
   
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    solve();
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self, heights: list[int]) -> int:
       
        heights.append(0)
        
        st = [] 
        max_area = 0
        
        for i in range(len(heights)):
            
            while st and heights[i] < heights[st[-1]]:
                h = heights[st.pop()]
                w = i if not st else i - st[-1] - 1
                max_area = max(max_area, h * w)
            
            st.append(i)
            
        return max_area

## J 消防局的设立

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

const int MAXN = 100005; 
int parent_node[MAXN];
int depth[MAXN];
vector<int> adj[MAXN];
bool covered[MAXN];

int main() {
   
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    if (!(cin >> n)) return 0;

    
    parent_node[1] = 1; 
    depth[1] = 1;

    
    for (int i = 2; i <= n; ++i) {
        cin >> parent_node[i];
        
       
        adj[i].push_back(parent_node[i]);
        adj[parent_node[i]].push_back(i);
        
        
        depth[i] = depth[parent_node[i]] + 1;
    }

    
    vector<int> order(n);
    for (int i = 0; i < n; ++i) {
        order[i] = i + 1;
    }

   
    sort(order.begin(), order.end(), [&](int x, int y) {
        return depth[x] > depth[y];
    });

    int ans = 0;
    
    
    for (int u : order) {
        if (!covered[u]) {
            
            int p = parent_node[u];
            int gp = parent_node[p];
            
            
            ans++;
            
            
            covered[gp] = true;
            for (int v : adj[gp]) {
                covered[v] = true; 
                for (int w : adj[v]) {
                    covered[w] = true; 
                }
            }
        }
    }

    cout << ans << "\n";

    return 0;
}